# Random Forest

In [1]:
import numpy as np
import umap
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import tensorflow as tf
import statsmodels.api as sm
from sklearn.svm import SVC
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import pearsonr, pointbiserialr, spearmanr
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve, auc, silhouette_score, precision_recall_curve, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, Concatenate
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

D:\anaconda3\envs\tensorflow\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 设置随机种子以确保结果的可重复性
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)

In [4]:
# 加载文本特征
#text_embeddings = np.load('PubMedBERT.npy')
text_embeddings = np.load('Bioformer.npy')

In [5]:
# scaler = StandardScaler()
# text_embeddings_scaled = scaler.fit_transform(text_embeddings)

# pca = PCA(n_components=10)
# text_features_pca = pca.fit_transform(text_embeddings_scaled)
text_features_pca = umap.UMAP(n_components=2, random_state=42).fit_transform(MinMaxScaler().fit_transform(text_embeddings))
# 动态生成列名
n_pca_features = text_features_pca.shape[1]
pca_columns = [f'Text_Embedding_Combined{i+1}' for i in range(n_pca_features)]

# 转换为DataFrame
text_feature_pca_df = pd.DataFrame(text_features_pca, columns=pca_columns)

In [6]:
X = df[['LN转移个数','腋窝淋巴结状态','PR', 'HER2+FISH', 'ki-67', 
        '手术前怀孕', '治疗后怀孕', '治疗后生产','目前月经情况',
        '手术方式','放疗', '化疗期间是否应用诺雷德', 
        '靶向治疗（赫赛汀或赫赛汀+帕捷特）','化疗方案', '内分泌治疗方案']]
#X = df[['LN转移个数','腋窝淋巴结状态','PR', 'HER2+FISH', 'ki-67', '手术前怀孕', '治疗后怀孕', '治疗后生产','目前月经情况','手术方式','放疗', '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','化疗方案', '内分泌治疗方案','Text_Embedding_Combined1','Text_Embedding_Combined2']]

In [7]:
X = pd.concat([X, text_feature_pca_df], axis=1)

In [8]:
X

,LN转移个数,腋窝淋巴结状态,PR,HER2+FISH,ki-67,手术前怀孕,治疗后怀孕,治疗后生产,目前月经情况,手术方式,放疗,化疗期间是否应用诺雷德,靶向治疗（赫赛汀或赫赛汀+帕捷特）,化疗方案,内分泌治疗方案,Text_Embedding_Combined1,Text_Embedding_Combined2
0,5,1,10,1,0.20,2,0,0,6,2,1,0,1,2,1,9.328062,10.847112
1,2,1,95,2,0.10,1,0,0,1,2,1,0,0,4,1,5.112262,10.129079
2,0,0,70,1,0.05,3,0,0,5,1,1,0,0,8,1,6.144903,12.094459
3,6,1,3,0,0.60,2,0,0,7,5,1,0,0,4,1,7.315619,10.285522
4,0,0,90,3,0.05,4,0,0,6,5,0,0,1,2,1,8.315938,12.810222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1361,0,0,90,0,0.40,2,0,0,2,1,1,0,0,0,1,3.628621,10.355165
1362,1,1,95,0,0.45,0,0,0,1,1,1,0,0,0,3,5.275845,10.167602
1363,8,1,0,1,0.40,1,0,0,1,4,1,0,0,7,0,7.094494,9.019923
1364,0,0,95,2,0.15,3,0,0,2,1,1,0,0,8,0,5.570846,13.239636


In [9]:
y = df['标签']

In [10]:
# 连续变量
#continuous_vars = ['PR', 'ki-67'] + pca_columns
continuous_vars = ['PR', 'ki-67','Text_Embedding_Combined1','Text_Embedding_Combined2']

In [11]:
# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

In [12]:
# # 使用分层划分以保持类别比例
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [13]:
# 2. 连续变量：进行标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])

In [14]:
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

In [15]:
# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

In [16]:
class_weight_dict

{0: 0.8136482939632546, 1: 1.297071129707113}

In [17]:
# 定义随机森林分类器
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

In [18]:
#定义超参数网格
param_grid = {
    'n_estimators': [100, 200, 300],  # 树的数量
    'max_depth': [None, 10, 20, 30],  # 树的最大深度
    'min_samples_split': [2, 5, 10],  # 内部节点再划分所需最小样本数
    'min_samples_leaf': [1, 2, 4],  # 叶子节点所需最小样本数
    'max_features': ['sqrt', 'log2', None],  # 划分时考虑的最大特征数
    'class_weight': [None, 'balanced', class_weight_dict]  # 类别权重选项
}

In [19]:
# 使用网格搜索和交叉验证来寻找最佳参数
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='roc_auc')
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 972 candidates, totalling 4860 fits


GridSearchCV(cv=5,
             estimator=RandomForestClassifier(class_weight='balanced',
                                              random_state=42),
             n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced',
                                          {0: 0.8136482939632546,
                                           1: 1.297071129707113}],
                         'max_depth': [None, 10, 20, 30],
                         'max_features': ['sqrt', 'log2', None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='roc_auc', verbose=2)

In [20]:
# 输出最佳参数和对应得分
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

Best parameters found:  {'class_weight': None, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}
Best cross-validation score:  0.911595072239422


In [21]:
# 使用最佳模型进行预测
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

In [22]:
y_test_proba = best_rf.predict_proba(X_test)[:, 1]
# 评估模型性能
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_test_proba)
print(f"Accuracy (Adjusted Threshold): {accuracy}")
print(f"F1 Score: {f1}")
print(f"ROC AUC: {roc_auc}")

Accuracy (Adjusted Threshold): 0.7063492063492064
F1 Score: 0.4126984126984127
ROC AUC: 0.7254901960784313


In [23]:
# 评估模型
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[76 26]
 [11 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.75      0.80       102
           1       0.33      0.54      0.41        24

    accuracy                           0.71       126
   macro avg       0.60      0.64      0.61       126
weighted avg       0.77      0.71      0.73       126



In [24]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("rf_yy_roc_curve_2.csv", index=False)
print("ROC数据已保存为 rf_yy_roc_curve_2.csv")

ROC数据已保存为 rf_yy_roc_curve_2.csv


In [25]:
import joblib

# 保存最优模型和标准化器
joblib.dump(best_rf, 'bf_w_2.pkl')
joblib.dump(scaler, 'scaler_bf_w_2.pkl')

print("✅ 模型和标准化器已保存。")

✅ 模型和标准化器已保存。


# XGBoost

In [26]:
import numpy as np
import umap
import pandas as pd
import xgboost as xgb
import seaborn as sns
import lightgbm as lgb
import tensorflow as tf
import statsmodels.api as sm
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import pearsonr, pointbiserialr, spearmanr
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve, auc, silhouette_score, precision_recall_curve, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, Concatenate
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [27]:
# 设置随机种子以确保结果的可重复性
np.random.seed(42)
tf.random.set_seed(42)

In [28]:
# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)
# 加载文本特征
#text_embeddings = np.load('PubMedBERT.npy')
text_embeddings = np.load('Bioformer_8L_meanpool.npy')
# scaler = StandardScaler()
# text_embeddings_scaled = scaler.fit_transform(text_embeddings)

# pca = PCA(n_components=2)
# text_features_pca = pca.fit_transform(text_embeddings_scaled)
text_features_pca = umap.UMAP(n_components=2, random_state=42).fit_transform(MinMaxScaler().fit_transform(text_embeddings))
# 动态生成列名
n_pca_features = text_features_pca.shape[1]
pca_columns = [f'Text_Embedding_Combined{i+1}' for i in range(n_pca_features)]

# 转换为DataFrame
text_feature_pca_df = pd.DataFrame(text_features_pca, columns=pca_columns)
X = df[['LN转移个数','腋窝淋巴结状态','PR', 'HER2+FISH', 'ki-67', '手术前怀孕', '治疗后怀孕', '治疗后生产','目前月经情况','手术方式','放疗', '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','化疗方案', '内分泌治疗方案']]
X = pd.concat([X, text_feature_pca_df], axis=1)
y = df['标签']
# 连续变量
continuous_vars = ['PR', 'ki-67'] + pca_columns

# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 2. 连续变量：进行标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])
# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

In [29]:
scale_pos_weight = class_weight_dict.get(1, 1)

In [30]:
# 定义 XGBoost 分类器
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic',  # 二分类逻辑回归
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

In [31]:
# 定义超参数网格
param_grid = {
    'n_estimators': [100, 200, 300],  # 树的数量
    'max_depth': [3, 5, 7],  # 树的最大深度
    'learning_rate': [0.01, 0.1, 0.2],  # 学习率
    'subsample': [0.7, 0.8, 1.0],  # 每次迭代时随机选择的样本比例
    'colsample_bytree': [0.5, 0.7, 1.0],  # 每棵树随机选择的特征比例
    'gamma': [0, 0.1, 0.2],  # 树的叶子节点上进行进一步分裂所需的最小损失减少量
    'min_child_weight': [1, 2, 3]  # 子节点中最小的样本权重和
}

In [32]:
# 使用网格搜索和交叉验证来寻找最佳参数
grid_search = GridSearchCV(estimator=xgb_classifier, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='roc_auc')
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 2187 candidates, totalling 10935 fits


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None,...
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None,
                                     random_state=42, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.5, 0.7, 1.0],
                         'gamma': [0, 0.1, 0.2],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7], 'min_child_weight': [1, 2, 3],
                         'n_estimators': [100, 200, 300],
                         'subsample': [0.7, 0.8, 1.0]},
             scoring='roc_auc', verbose=2)

In [33]:
# 输出最佳参数和对应得分
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

Best parameters found:  {'colsample_bytree': 0.5, 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 1.0}
Best cross-validation score:  0.9168100167698657


In [34]:
# 获取最佳模型
best_xgb = grid_search.best_estimator_

In [35]:
# 使用最佳模型进行预测
y_pred = best_xgb.predict(X_test)

In [36]:
y_test_proba = best_xgb.predict_proba(X_test)[:, 1]
# 评估模型性能
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_test_proba)
print(f"Accuracy (Adjusted Threshold): {accuracy}")
print(f"F1 Score: {f1}")
print(f"ROC AUC: {roc_auc}")

Accuracy (Adjusted Threshold): 0.7142857142857143
F1 Score: 0.41935483870967744
ROC AUC: 0.6977124183006536


In [37]:
# 计算 FPR, TPR, Thresholds
from sklearn.metrics import roc_curve
import pandas as pd

fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

# 保存为CSV，供画图使用
roc_df = pd.DataFrame({
    'FPR': fpr,
    'TPR': tpr,
    'Threshold': thresholds
})
roc_df.to_csv("xgb_yy_roc_curve_2.csv", index=False)
print("ROC 曲线数据已保存至 xgb_yy_roc_curve_2.csv")

ROC 曲线数据已保存至 xgb_yy_roc_curve_2.csv


In [38]:
import joblib

# 保存最优模型和标准化器
joblib.dump(best_xgb, 'best_xgb_2.pkl')
joblib.dump(scaler, 'scaler_xgb_2.pkl')

print("✅ 模型和标准化器已保存。")

✅ 模型和标准化器已保存。


# Gradient Boosting Trees

In [39]:
import numpy as np
import umap
import pandas as pd
import xgboost as xgb
import seaborn as sns
import lightgbm as lgb
import tensorflow as tf
import statsmodels.api as sm
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import pearsonr, pointbiserialr, spearmanr
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve, auc, silhouette_score, precision_recall_curve, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, Concatenate
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [40]:
# 设置随机种子以确保结果的可重复性
np.random.seed(42)
tf.random.set_seed(42)

In [41]:
# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)
# 加载文本特征
#text_embeddings = np.load('PubMedBERT.npy')
text_embeddings = np.load('Bioformer.npy')
# scaler = StandardScaler()
# text_embeddings_scaled = scaler.fit_transform(text_embeddings)

# pca = PCA(n_components=2)
# text_features_pca = pca.fit_transform(text_embeddings_scaled)
text_features_pca = umap.UMAP(n_components=2, random_state=42).fit_transform(MinMaxScaler().fit_transform(text_embeddings))
# 动态生成列名
n_pca_features = text_features_pca.shape[1]
pca_columns = [f'Text_Embedding_Combined{i+1}' for i in range(n_pca_features)]

# 转换为DataFrame
text_feature_pca_df = pd.DataFrame(text_features_pca, columns=pca_columns)
X = df[['LN转移个数','腋窝淋巴结状态','PR', 'HER2+FISH', 'ki-67', '手术前怀孕', '治疗后怀孕', '治疗后生产','目前月经情况','手术方式','放疗', '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','化疗方案', '内分泌治疗方案']]
X = pd.concat([X, text_feature_pca_df], axis=1)
y = df['标签']
# 连续变量
continuous_vars = ['PR', 'ki-67'] + pca_columns
# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 2. 连续变量：进行标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])
# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

In [42]:
sample_weights_train = np.array([class_weight_dict[label] for label in y_train])

In [43]:
# 定义梯度提升树分类器
gbc = GradientBoostingClassifier(random_state=42)

In [44]:
# 定义超参数网格
param_grid = {
    'n_estimators': [50, 100, 200],  # 树的数量
    'learning_rate': [0.01, 0.1, 0.2],  # 学习率
    'max_depth': [3, 5, 7],  # 树的最大深度
    'min_samples_split': [2, 5, 10],  # 内部节点再划分所需最小样本数
    'min_samples_leaf': [1, 2, 4],  # 叶子节点所需最小样本数
    'subsample': [0.7, 0.8, 1.0]  # 每次迭代时随机选择的样本比例
}

In [45]:
# 使用网格搜索和交叉验证来寻找最佳参数
grid_search = GridSearchCV(estimator=gbc, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='roc_auc')
grid_search.fit(X_train, y_train, sample_weight=sample_weights_train)

Fitting 5 folds for each of 729 candidates, totalling 3645 fits


GridSearchCV(cv=5, estimator=GradientBoostingClassifier(random_state=42),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7], 'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200],
                         'subsample': [0.7, 0.8, 1.0]},
             scoring='roc_auc', verbose=2)

In [46]:
# 输出最佳参数和对应得分
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

Best parameters found:  {'learning_rate': 0.01, 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200, 'subsample': 0.7}
Best cross-validation score:  0.915047568369453


In [47]:
best_gbc = grid_search.best_estimator_

In [48]:
# 使用最佳模型进行预测
y_pred = best_gbc.predict(X_test)

In [49]:
y_test_proba = best_gbc.predict_proba(X_test)[:, 1]
# 评估模型性能
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_test_proba)
print(f"Accuracy (Adjusted Threshold): {accuracy}")
print(f"F1 Score: {f1}")
print(f"ROC AUC: {roc_auc}")

Accuracy (Adjusted Threshold): 0.7222222222222222
F1 Score: 0.4262295081967213
ROC AUC: 0.7107843137254901


In [50]:
# 评估模型
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[78 24]
 [11 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.76      0.82       102
           1       0.35      0.54      0.43        24

    accuracy                           0.72       126
   macro avg       0.61      0.65      0.62       126
weighted avg       0.78      0.72      0.74       126



In [51]:
# 计算 FPR, TPR, Thresholds
from sklearn.metrics import roc_curve
import pandas as pd

fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

# 保存为CSV，供画图使用
roc_df = pd.DataFrame({
    'FPR': fpr,
    'TPR': tpr,
    'Threshold': thresholds
})
roc_df.to_csv("gbt_yy_roc_curve_2.csv", index=False)
print("ROC 曲线数据已保存至 gbt_yy_roc_curve_2.csv")

ROC 曲线数据已保存至 gbt_yy_roc_curve_2.csv


In [52]:
import joblib

# 保存最优模型和标准化器
joblib.dump(best_gbc, 'best_gbc_2.pkl')
joblib.dump(scaler, 'scaler_gbc_2.pkl')

print("✅ 模型和标准化器已保存。")

✅ 模型和标准化器已保存。


# LightGBM

In [53]:
import numpy as np
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import tensorflow as tf
import statsmodels.api as sm
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import pearsonr, pointbiserialr, spearmanr
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve, auc, silhouette_score, precision_recall_curve, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, Concatenate
import warnings
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import umap
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [54]:
# 设置随机种子以确保结果的可重复性
np.random.seed(42)
tf.random.set_seed(42)

In [55]:
# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)
# 加载文本特征
#text_embeddings = np.load('PubMedBERT.npy')
text_embeddings = np.load('Bioformer.npy')
# scaler = StandardScaler()
# text_embeddings_scaled = scaler.fit_transform(text_embeddings)

# pca = PCA(n_components=2)
# text_features_pca = pca.fit_transform(text_embeddings_scaled)
text_features_pca = umap.UMAP(n_components=2, random_state=42).fit_transform(MinMaxScaler().fit_transform(text_embeddings))
# 动态生成列名
n_pca_features = text_features_pca.shape[1]
pca_columns = [f'Text_Embedding_Combined{i+1}' for i in range(n_pca_features)]

# 转换为DataFrame
text_feature_pca_df = pd.DataFrame(text_features_pca, columns=pca_columns)
X = df[['LN转移个数','腋窝淋巴结状态','PR', 'HER2+FISH', 'ki-67', '手术前怀孕', '治疗后怀孕', '治疗后生产','目前月经情况','手术方式','放疗', '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','化疗方案', '内分泌治疗方案']]
X = pd.concat([X, text_feature_pca_df], axis=1)
y = df['标签']
# 连续变量
continuous_vars = ['PR', 'ki-67'] + pca_columns
# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 2. 连续变量：进行标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

In [56]:
# 计算类别权重（LightGBM需要手动计算正样本权重）
positive_count = np.sum(y_train == 1)
negative_count = np.sum(y_train == 0)
scale_pos_weight = negative_count / positive_count  # 用于处理不平衡数据

In [57]:
# 定义LightGBM分类器
lgbm = lgb.LGBMClassifier(
    random_state=42,
    objective='binary',
    metric='auc',
    n_jobs=-1,  # 使用所有CPU核心
    verbose=-1,  # 不输出训练日志
    scale_pos_weight=scale_pos_weight  # 处理类别不平衡
)

In [58]:
# 定义LightGBM的超参数网格
param_grid = {
    'n_estimators': [50, 100, 150],  # 树的数量
    'num_leaves': [15, 30, 45],  # 最大叶子数
    'max_depth': [3, 5, 7],  # 树的最大深度（-1表示无限制）
    'learning_rate': [0.01, 0.05],  # 学习率
    'min_child_samples': [20, 30, 40],  # 叶子节点最小样本数
    'subsample': [0.7, 0.8],  # 样本采样比例
    'colsample_bytree': [0.6, 0.7],  # 特征采样比例
    'reg_alpha': [0.1, 0.5, 1.0],  # L1正则化
    'reg_lambda': [0.1, 0.5, 1.0],  # L2正则化
}

In [59]:
# 使用网格搜索和交叉验证来寻找最佳参数
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='roc_auc'
)

In [60]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 5832 candidates, totalling 29160 fits


GridSearchCV(cv=5,
             estimator=LGBMClassifier(metric='auc', n_jobs=-1,
                                      objective='binary', random_state=42,
                                      scale_pos_weight=1.594142259414226,
                                      verbose=-1),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.6, 0.7],
                         'learning_rate': [0.01, 0.05], 'max_depth': [3, 5, 7],
                         'min_child_samples': [20, 30, 40],
                         'n_estimators': [50, 100, 150],
                         'num_leaves': [15, 30, 45],
                         'reg_alpha': [0.1, 0.5, 1.0],
                         'reg_lambda': [0.1, 0.5, 1.0],
                         'subsample': [0.7, 0.8]},
             scoring='roc_auc', verbose=2)

In [61]:
# 使用最佳模型进行预测
best_lgbm = grid_search.best_estimator_
y_pred = best_lgbm.predict(X_test)
y_test_proba = best_lgbm.predict_proba(X_test)[:, 1]

In [62]:
# 评估模型性能
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_test_proba)

In [63]:
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
# 评估模型
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_samples': 20, 'n_estimators': 100, 'num_leaves': 15, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'subsample': 0.7}
Accuracy: 0.7143
F1 Score: 0.4194
ROC AUC: 0.7390

Confusion Matrix:
[[77 25]
 [11 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.75      0.81       102
           1       0.34      0.54      0.42        24

    accuracy                           0.71       126
   macro avg       0.61      0.65      0.61       126
weighted avg       0.77      0.71      0.74       126



In [64]:
# 计算 FPR, TPR, Thresholds
from sklearn.metrics import roc_curve
import pandas as pd

fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

# 保存为CSV，供画图使用
roc_df = pd.DataFrame({
    'FPR': fpr,
    'TPR': tpr,
    'Threshold': thresholds
})
roc_df.to_csv("lgbm_yy_roc_curve_2.csv", index=False)
print("ROC 曲线数据已保存至 lgbm_yy_roc_curve_2.csv")

ROC 曲线数据已保存至 lgbm_yy_roc_curve_2.csv


In [65]:
import joblib

# 保存最优模型和标准化器
joblib.dump(best_lgbm, 'best_lgbm_2.pkl')
joblib.dump(scaler, 'scaler_lgbm_2.pkl')

print("✅ 模型和标准化器已保存。")

✅ 模型和标准化器已保存。


# CatBoost

In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, 
                             confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_sample_weight
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import umap

In [67]:
# 设置随机种子
np.random.seed(42)

In [68]:
# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)
# 加载文本特征
#text_embeddings = np.load('PubMedBERT.npy')
text_embeddings = np.load('Bioformer.npy')
# scaler = StandardScaler()
# text_embeddings_scaled = scaler.fit_transform(text_embeddings)

# pca = PCA(n_components=2)
# text_features_pca = pca.fit_transform(text_embeddings_scaled)
text_features_pca = umap.UMAP(n_components=2, random_state=42).fit_transform(MinMaxScaler().fit_transform(text_embeddings))
# 动态生成列名
n_pca_features = text_features_pca.shape[1]
pca_columns = [f'Text_Embedding_Combined{i+1}' for i in range(n_pca_features)]

# 转换为DataFrame
text_feature_pca_df = pd.DataFrame(text_features_pca, columns=pca_columns)
X = df[['LN转移个数','腋窝淋巴结状态','PR', 'HER2+FISH', 'ki-67', '手术前怀孕', '治疗后怀孕', '治疗后生产','目前月经情况','手术方式','放疗', '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','化疗方案', '内分泌治疗方案']]
X = pd.concat([X, text_feature_pca_df], axis=1)
y = df['标签']
# 连续变量
continuous_vars = ['PR', 'ki-67'] + pca_columns
# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)
# 连续变量：进行标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

In [69]:
# 5. 类别样本权重
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

In [70]:
# 6. CatBoost 参数网格
param_grid = {
    'iterations': [100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'depth': [4, 6],
    'l2_leaf_reg': [1, 3, 5],
    'border_count': [32, 64],  # 用于数值特征的分箱
}

In [71]:
# 7. 定义模型
cat = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0,
    random_seed=42
)

In [72]:
# 8. 网格搜索 + 分层K折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=cat,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    scoring='roc_auc'
)

In [73]:
# 9. 拟合模型
grid_search.fit(X_train, y_train, sample_weight=sample_weights)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=<catboost.core.CatBoostClassifier object at 0x0000020342EEB400>,
             n_jobs=-1,
             param_grid={'border_count': [32, 64], 'depth': [4, 6],
                         'iterations': [100, 200], 'l2_leaf_reg': [1, 3, 5],
                         'learning_rate': [0.05, 0.1, 0.2]},
             scoring='roc_auc', verbose=1)

In [74]:
# 10. 预测与评估
best_cat = grid_search.best_estimator_
y_pred = best_cat.predict(X_test)
y_test_proba = best_cat.predict_proba(X_test)[:, 1]

In [75]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_test_proba)

In [76]:
# 11. 输出结果
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Best Parameters: {'border_count': 64, 'depth': 6, 'iterations': 100, 'l2_leaf_reg': 3, 'learning_rate': 0.2}
Accuracy: 0.7063
F1 Score: 0.4127
ROC AUC: 0.7222

Confusion Matrix:
[[76 26]
 [11 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.75      0.80       102
           1       0.33      0.54      0.41        24

    accuracy                           0.71       126
   macro avg       0.60      0.64      0.61       126
weighted avg       0.77      0.71      0.73       126



In [77]:
# 计算 FPR, TPR, Thresholds
from sklearn.metrics import roc_curve
import pandas as pd

fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

# 保存为CSV，供画图使用
roc_df = pd.DataFrame({
    'FPR': fpr,
    'TPR': tpr,
    'Threshold': thresholds
})
roc_df.to_csv("catb_yy_roc_curve_2.csv", index=False)
print("ROC 曲线数据已保存至 catb_yy_roc_curve_2.csv")

ROC 曲线数据已保存至 catb_yy_roc_curve_2.csv


In [78]:
import joblib

# 保存最优模型和标准化器
joblib.dump(best_cat, 'best_cat_2.pkl')
joblib.dump(scaler, 'scaler_cat_2.pkl')

print("✅ 模型和标准化器已保存。")

✅ 模型和标准化器已保存。
